# Day 006 — Circular Queue · Quick Sort · Fibonacci Search
**Date:** 2026-08-17  |  **Difficulty:** Beginner-Intermediate  |  **Series:** Daily DSA

---

## What You Will Learn Today
| Topic | Concept | Time Complexity |
|-------|---------|----------------|
| Data Structure | **Circular Queue** | Enqueue/Dequeue O(1) |
| Sorting | **Quick Sort** | Best/Avg O(n log n), Worst O(n²) |
| Searching | **Fibonacci Search** | O(log n) — works on sorted arrays |

> **Series recap**
> - Day 001: Array · Bubble Sort · Linear Search
> - Day 002: Singly Linked List · Selection Sort · Jump Search
> - Day 003: Doubly Linked List · Insertion Sort · Binary Search
> - Day 004: Stack · Shell Sort · Sentinel Search
> - Day 005: Queue · Merge Sort · Interpolation Search
> - **Today**: Circular Queue (fixed-size ring buffer) · Quick Sort (in-place partition) · Fibonacci Search

> Run each cell top-to-bottom with **Shift+Enter** to follow along interactively.

---
## PART 1 — Data Structure: Circular Queue

A **Circular Queue** (also called a **Ring Buffer**) is a fixed-size queue where the
rear wraps around to the front when it reaches the end of the underlying array.
This reuses freed slots and keeps both enqueue and dequeue at **O(1)**.

### Visual (capacity = 5)
```
Index:   0    1    2    3    4
        [ A ][ B ][ C ][   ][   ]
          ↑              ↑
         front          rear

After dequeue(A) and enqueue(D):
        [   ][ B ][ C ][ D ][   ]
               ↑         ↑
             front       rear

After enqueue(E), enqueue(F):
        [ F ][ B ][ C ][ D ][ E ]
               ↑    ↑
             front  rear  (wraps around!)
```

### Core operations — all O(1)
| Operation | Description |
|-----------|-------------|
| `enqueue(x)` | Add to rear (wraps if needed) |
| `dequeue()` | Remove from front (wraps if needed) |
| `peek()` | Read front without removing |
| `is_full()` | True when size == capacity |
| `is_empty()` | True when size == 0 |

### Why use a Circular Queue over a plain Queue?
- Plain list-based queue: `pop(0)` is **O(n)** — shifts every element left
- `collections.deque`: great general purpose, but **unbounded** by default
- Circular Queue: **O(1)** both ends, **fixed memory**, no allocations after init
- Used in: OS scheduling, audio/video streaming buffers, network packet queues

In [ ]:
# ─── Circular Queue Implementation ───────────────────────────────────────────

class CircularQueue:
    """
    Fixed-capacity FIFO queue backed by a ring buffer.
    Enqueue and dequeue are both O(1) — no shifting, no reallocation.

    The 'size' counter distinguishes full from empty:
    - empty: size == 0
    - full:  size == capacity
    This avoids the classic off-by-one confusion of the (front == rear) ambiguity.
    """

    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError('capacity must be >= 1')
        self._buf      = [None] * capacity
        self._capacity = capacity
        self._front    = 0
        self._rear     = 0
        self._size     = 0

    def enqueue(self, item):
        """Add item to the rear — O(1)."""
        if self.is_full():
            raise OverflowError(f'CircularQueue is full (capacity={self._capacity})')
        self._buf[self._rear] = item
        self._rear = (self._rear + 1) % self._capacity
        self._size += 1

    def dequeue(self):
        """Remove and return the front item — O(1)."""
        if self.is_empty():
            raise IndexError('dequeue from empty CircularQueue')
        item = self._buf[self._front]
        self._buf[self._front] = None       # help GC
        self._front = (self._front + 1) % self._capacity
        self._size -= 1
        return item

    def peek(self):
        """Return the front item without removing it — O(1)."""
        if self.is_empty():
            raise IndexError('peek at empty CircularQueue')
        return self._buf[self._front]

    def is_empty(self): return self._size == 0
    def is_full(self):  return self._size == self._capacity
    def size(self):     return self._size

    def to_list(self):
        """Return items front-to-rear as a plain list — O(n)."""
        result = []
        idx = self._front
        for _ in range(self._size):
            result.append(self._buf[idx])
            idx = (idx + 1) % self._capacity
        return result

    def __repr__(self):
        buf_view = [str(x) if x is not None else '_' for x in self._buf]
        markers  = [' '] * self._capacity
        if not self.is_empty():
            markers[self._front] = 'F'
            markers[(self._rear - 1) % self._capacity] = 'R'
            if self._front == (self._rear - 1) % self._capacity:
                markers[self._front] = 'B'  # both front and rear
        slots = ' | '.join(f'{m}{v}' for m, v in zip(markers, buf_view))
        return f'CQ[{slots}]  size={self._size}/{self._capacity}'


# ── Demo ──────────────────────────────────────────────────────────────────────
cq = CircularQueue(5)
print('=== Enqueue A, B, C ===')
for ch in ['A', 'B', 'C']:
    cq.enqueue(ch)
    print(f'  enqueue({ch!r}) → {cq}')

print(f'\npeek() → {cq.peek()!r}')

print('\n=== Dequeue two ===')
for _ in range(2):
    print(f'  dequeue() → {cq.dequeue()!r}  | {cq}')

print('\n=== Enqueue D, E, F (wrap-around) ===')
for ch in ['D', 'E', 'F']:
    cq.enqueue(ch)
    print(f'  enqueue({ch!r}) → {cq}')

print(f'\nis_full()  → {cq.is_full()}')
print(f'to_list()  → {cq.to_list()}')

print('\n=== Drain queue ===')
while not cq.is_empty():
    print(f'  dequeue() → {cq.dequeue()!r}  | {cq}')

In [ ]:
# ─── Edge-case tests ─────────────────────────────────────────────────────────

print('--- Edge Cases ---')

# Overflow
cq2 = CircularQueue(2)
cq2.enqueue(1); cq2.enqueue(2)
try:
    cq2.enqueue(3)
    print('  ✗ Should have raised OverflowError')
except OverflowError as e:
    print(f'  ✓ OverflowError on full queue: {e}')

# Underflow
cq3 = CircularQueue(3)
try:
    cq3.dequeue()
    print('  ✗ Should have raised IndexError')
except IndexError as e:
    print(f'  ✓ IndexError on empty queue: {e}')

# Wrap-around correctness: fill → drain → refill
cq4 = CircularQueue(3)
for v in [10, 20, 30]: cq4.enqueue(v)
cq4.dequeue(); cq4.dequeue()           # free two slots at the front
cq4.enqueue(40); cq4.enqueue(50)       # these wrap around to index 0 and 1
result = cq4.to_list()
assert result == [30, 40, 50], f'Expected [30,40,50], got {result}'
print(f'  ✓ Wrap-around correct: {result}')

# Capacity 1
cq5 = CircularQueue(1)
cq5.enqueue('X')
assert cq5.dequeue() == 'X'
cq5.enqueue('Y')
assert cq5.peek() == 'Y'
print(f'  ✓ Capacity-1 queue works')

print('\nAll edge cases passed.')

---
## PART 2 — Sorting Algorithm: Quick Sort

**Quick Sort** is a divide-and-conquer algorithm that picks a **pivot**, partitions the
array into elements ≤ pivot and elements > pivot, then recursively sorts each side.

### Lomuto vs Hoare partition
| Scheme | Pivot position | Swaps (avg) | Notes |
|--------|---------------|-------------|-------|
| Lomuto | last element | ~n²/4 | simpler to implement |
| Hoare | first element | ~n²/6 | fewer swaps, used here |

### Complexity
| Case | Time | Space |
|------|------|-------|
| Best | O(n log n) — balanced partitions | O(log n) call stack |
| Average | O(n log n) | O(log n) |
| Worst | O(n²) — already sorted + bad pivot | O(n) |

### Worst-case avoidance
- **Median-of-three**: pick median of `arr[lo]`, `arr[mid]`, `arr[hi]` as pivot
- **Random pivot**: shuffle or pick random index — O(n²) becomes astronomically unlikely
- **3-way partition** (Dutch National Flag): handles many duplicates efficiently

We implement **Lomuto with median-of-three** pivot — easy to follow and avoids worst case.

In [ ]:
# ─── Quick Sort (Lomuto partition, median-of-three pivot) ─────────────────────

def _median_of_three(arr, lo, hi):
    """Return index of the median of arr[lo], arr[mid], arr[hi]."""
    mid = (lo + hi) // 2
    a, b, c = arr[lo], arr[mid], arr[hi]
    if (a <= b <= c) or (c <= b <= a): return mid
    if (b <= a <= c) or (c <= a <= b): return lo
    return hi


def _partition(arr, lo, hi, counters):
    """
    Lomuto partition: place pivot at arr[hi], return its final index.
    Pivot is chosen via median-of-three and moved to arr[hi] first.
    """
    pivot_idx = _median_of_three(arr, lo, hi)
    arr[pivot_idx], arr[hi] = arr[hi], arr[pivot_idx]   # move pivot to end
    pivot = arr[hi]
    i = lo - 1
    for j in range(lo, hi):
        counters['comparisons'] += 1
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
            counters['swaps'] += 1
    arr[i + 1], arr[hi] = arr[hi], arr[i + 1]
    counters['swaps'] += 1
    return i + 1


def _quick_sort_recursive(arr, lo, hi, counters, depth, verbose):
    if lo >= hi:
        return
    p = _partition(arr, lo, hi, counters)
    if verbose:
        indent = '  ' * depth
        print(f'{indent}pivot={arr[p]}  left={arr[lo:p]}  right={arr[p+1:hi+1]}')
    _quick_sort_recursive(arr, lo, p - 1, counters, depth + 1, verbose)
    _quick_sort_recursive(arr, p + 1, hi, counters, depth + 1, verbose)


def quick_sort(arr, verbose=False):
    """
    In-place Quick Sort — returns (sorted_list, swaps, comparisons).
    Works on a copy to leave the original unchanged.
    """
    a = arr[:]
    counters = {'swaps': 0, 'comparisons': 0}
    if len(a) > 1:
        _quick_sort_recursive(a, 0, len(a) - 1, counters, 0, verbose)
    return a, counters['swaps'], counters['comparisons']


sample = [38, 27, 43, 3, 9, 82, 10]
print(f'Input : {sample}\n')
print('--- Partition trace ---')
sorted_arr, swaps, comps = quick_sort(sample, verbose=True)
print(f'\nResult: {sorted_arr}')
print(f'Swaps: {swaps}  |  Comparisons: {comps}')

In [ ]:
# ─── Quick Sort test suite ────────────────────────────────────────────────────

test_cases = [
    ([38, 27, 43, 3, 9, 82, 10],  'random'),
    ([1, 2, 3, 4, 5],             'already sorted'),
    ([5, 4, 3, 2, 1],             'reverse sorted'),
    ([42],                        'single element'),
    ([],                          'empty list'),
    ([3, 3, 1, 1, 2, 2],          'many duplicates'),
    ([-7, 0, 5, -3, 8, -1],       'with negatives'),
    (list(range(15, 0, -1)),      'reverse 1-15'),
    ([1] * 10,                    'all identical'),
]

print(f'{"Input":<35} {"Sorted":<35} {"Swaps":>5} {"Cmps":>5}  Case')
print('-' * 95)
for data, label in test_cases:
    result, sw, cm = quick_sort(data)
    assert result == sorted(data), f'FAIL on {label}: {result} != {sorted(data)}'
    print(f'{str(data):<35} {str(result):<35} {sw:>5} {cm:>5}  {label}')

print('\nAll tests passed ✓')

---
## PART 3 — Searching Algorithm: Fibonacci Search

**Fibonacci Search** is a comparison-based search for **sorted arrays** that uses
Fibonacci numbers to divide the array — avoiding division entirely (only addition/subtraction).

### How it works
1. Find the smallest Fibonacci number ≥ n (the array length).
2. Split the array at `offset + fib(k-2)` — the split point from the Fibonacci sequence.
3. Compare target with that element:
   - Equal → found
   - Target smaller → move two steps down the Fibonacci sequence, search left
   - Target larger → move one step down, shift offset right, search right
4. Repeat until found or exhausted.

### Fibonacci numbers used
```
F(0)=0, F(1)=1, F(2)=1, F(3)=2, F(4)=3, F(5)=5,
F(6)=8, F(7)=13, F(8)=21, F(9)=34, F(10)=55 ...
```

### Complexity
| Case | Time | Space |
|------|------|-------|
| Best | O(1) | O(1) |
| Average / Worst | O(log n) | O(1) |

### Binary vs Fibonacci
- Binary Search splits at exact midpoint (requires division or bit shift)
- Fibonacci Search splits at Fibonacci-weighted offsets — useful on hardware where division is slow
- Both are O(log n); Fibonacci has a slightly larger constant but avoids division entirely

In [ ]:
# ─── Fibonacci Search ─────────────────────────────────────────────────────────

def fibonacci_search(arr, target, verbose=False):
    """
    Search for target in sorted arr using Fibonacci numbers.
    Returns the index of target, or -1 if not found.
    Array must be sorted in ascending order.
    """
    n = len(arr)
    if n == 0:
        return -1

    # Build up Fibonacci numbers until fib_m >= n
    fib_m2 = 0   # F(k-2)
    fib_m1 = 1   # F(k-1)
    fib_m  = 1   # F(k)
    while fib_m < n:
        fib_m2 = fib_m1
        fib_m1 = fib_m
        fib_m  = fib_m2 + fib_m1

    offset = -1   # marks the left boundary (elements before offset+1 are eliminated)
    step   = 0

    while fib_m > 1:
        # Valid index to compare — clamp to last element
        i = min(offset + fib_m2, n - 1)
        step += 1
        if verbose:
            print(f'  Step {step}: fib_m={fib_m}, fib_m2={fib_m2}, '
                  f'compare index {i} (value={arr[i]}) with target={target}')

        if arr[i] < target:
            # Eliminate left portion (including i)
            fib_m  = fib_m1
            fib_m1 = fib_m2
            fib_m2 = fib_m - fib_m1
            offset = i
            if verbose: print(f'          → target > arr[{i}], move right, offset={offset}')
        elif arr[i] > target:
            # Eliminate right portion
            fib_m  = fib_m2
            fib_m1 = fib_m1 - fib_m2
            fib_m2 = fib_m - fib_m1
            if verbose: print(f'          → target < arr[{i}], move left')
        else:
            if verbose: print(f'          → FOUND at index {i}')
            return i

    # Check one remaining element
    if fib_m1 and offset + 1 < n and arr[offset + 1] == target:
        if verbose: print(f'  Final check: index {offset+1} → FOUND')
        return offset + 1

    return -1


data = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
print(f'Array: {data}\n')

print('Search for 13 (present):')
idx = fibonacci_search(data, 13, verbose=True)
print(f'Result: index {idx} → value {data[idx] if idx != -1 else "not found"}\n')

print('Search for 6 (absent):')
idx = fibonacci_search(data, 6, verbose=True)
print(f'Result: {idx} (not found)')

In [ ]:
# ─── Fibonacci Search test suite ──────────────────────────────────────────────

sorted_data = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]

search_tests = [
    (sorted_data, 1,   'first element'),
    (sorted_data, 19,  'last element'),
    (sorted_data, 11,  'middle element'),
    (sorted_data, 6,   'not found — between elements'),
    (sorted_data, 0,   'not found — below range'),
    (sorted_data, 20,  'not found — above range'),
    ([42],         42, 'single element — found'),
    ([42],         7,  'single element — not found'),
    ([],           5,  'empty array'),
    ([2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22], 14, 'found in 11-element array'),
]

print(f'{"Array":<40} {"Target":>7}  {"Result":<14} Case')
print('-' * 80)
for arr, target, label in search_tests:
    result = fibonacci_search(arr, target)
    found  = f'index {result}' if result != -1 else 'not found'
    # Verify against Python's list.index
    expected = arr.index(target) if target in arr else -1
    ok = '✓' if result == expected else f'✗ (expected {expected})'
    print(f'{str(arr):<40} {str(target):>7}  {found:<14} {ok}  {label}')

print('\nAll tests passed ✓')

---
## PART 4 — End-to-End: OS Process Scheduler

Simulate a simplified OS scheduler:
1. **Circular Queue** holds arriving processes (fixed CPU time slice)
2. **Quick Sort** orders completed processes by burst time
3. **Fibonacci Search** finds a process by its burst time in the sorted list

In [ ]:
# ─── OS Process Scheduler simulation ─────────────────────────────────────────

processes = [
    {'pid': 'P1', 'burst': 8},
    {'pid': 'P2', 'burst': 3},
    {'pid': 'P3', 'burst': 12},
    {'pid': 'P4', 'burst': 5},
    {'pid': 'P5', 'burst': 7},
    {'pid': 'P6', 'burst': 2},
]

TIME_SLICE = 4   # CPU time slice per turn (Round-Robin)
CAPACITY   = 6

print('=== Step 1: Load processes into Circular Queue ===')
scheduler = CircularQueue(CAPACITY)
for p in processes:
    scheduler.enqueue(p)
    print(f'  enqueue({p["pid"]} burst={p["burst"]})')

print(f'\nQueue contents: {[p["pid"] for p in scheduler.to_list()]}')

print('\n=== Step 2: Round-Robin scheduling (time slice = {}) ==='.format(TIME_SLICE))
completed = []
time = 0
# drain and re-enqueue until all finish
remaining = list(processes)
rr_queue  = CircularQueue(CAPACITY)
for p in remaining:
    rr_queue.enqueue({'pid': p['pid'], 'burst_left': p['burst'], 'burst': p['burst']})

while not rr_queue.is_empty():
    proc = rr_queue.dequeue()
    run  = min(TIME_SLICE, proc['burst_left'])
    proc['burst_left'] -= run
    time += run
    if proc['burst_left'] == 0:
        proc['finish_time'] = time
        proc['turnaround']  = time
        completed.append(proc)
        print(f'  t={time:>3}: {proc["pid"]} DONE  (burst={proc["burst"]})')
    else:
        rr_queue.enqueue(proc)  # re-enqueue (wraps around)
        print(f'  t={time:>3}: {proc["pid"]} preempted, {proc["burst_left"]} left — re-queued')

print(f'\nTotal time: {time} units')

print('\n=== Step 3: Quick Sort completed processes by burst time ===')
burst_times = [p['burst'] for p in completed]
sorted_bursts, swaps, comps = quick_sort(burst_times)
print(f'Unsorted burst times : {burst_times}')
print(f'Sorted  burst times  : {sorted_bursts}  ({swaps} swaps, {comps} comparisons)')

print('\n=== Step 4: Fibonacci Search for burst time 7 ===')
target_burst = 7
pos = fibonacci_search(sorted_bursts, target_burst, verbose=True)
if pos != -1:
    print(f'\nFound burst={target_burst} at sorted index {pos}')
else:
    print(f'\nBurst={target_burst} not found in sorted list')

---
## Complexity Cheat Sheet

```
Circular Queue enqueue/dequeue/peek    O(1) — ring buffer, no shifting
Circular Queue is_full / is_empty      O(1)

Quick Sort (avg)                       O(n log n) — median-of-three pivot
Quick Sort (worst)                     O(n²)      — degenerate partitions
Quick Sort space                       O(log n)   — recursion call stack

Fibonacci Search                       O(log n)   — no division needed
Fibonacci Search (best)                O(1)       — first comparison hits target
```

**Key insight:** Quick Sort is the fastest in practice for general data despite its
O(n²) worst case — cache locality and low constant factor beat Merge Sort on
average-case inputs. Median-of-three pivot makes the worst case extremely rare.

**Tomorrow — Day 007:** Binary Tree · Heap Sort · Exponential Search